In [ ]:
import os
import subprocess
import sys

# ==================== Configuration (Adjust as needed) ====================

# 1. FFmpeg bin directory path (Set to None if FFmpeg is in system environment variables)
FFMPEG_BIN_PATH = r'C:\Users\PC\Downloads\ffmpeg-2026-03-12-git-9dc44b43b2-full_build\bin'

# 2. Download save directory (Defaults to Desktop)
OUTPUT_DIR = os.path.join(os.path.expanduser("~"), "Desktop", "BilibiliDownloads")

# 3. Video URL list (Supports BV IDs, webpage links, collections, etc.)
VIDEO_LIST = [
    "https://www.bilibili.com/video/BV1DxPPzdE9o", 
]

# 4. Video cropping parameters (Current: Keep bottom 980px, crop top 100px)
# Format: 'crop=width:height:x:y'
CROP_FILTER = 'crop=1920:980:0:100'

# ===========================================================================

def check_ffmpeg(ffmpeg_path=None):
    """
    Check if FFmpeg is installed and accessible.
    """
    ffmpeg_exe = 'ffmpeg'
    if ffmpeg_path:
        ffmpeg_exe = os.path.join(ffmpeg_path, 'ffmpeg')
        if sys.platform == 'win32':
            ffmpeg_exe += '.exe'
            
    try:
        subprocess.run([ffmpeg_exe, '-version'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return True
    except FileNotFoundError:
        return False

def crop_video(input_path, ffmpeg_bin_path=None):
    """
    Use FFmpeg to crop the video based on CROP_FILTER.
    """
    ffmpeg_exe = 'ffmpeg'
    if ffmpeg_bin_path:
        ffmpeg_exe = os.path.join(ffmpeg_bin_path, 'ffmpeg')
        if sys.platform == 'win32':
            ffmpeg_exe += '.exe'

    temp_output = input_path.replace('.mp4', '_cropped.mp4')
    
    crop_cmd = [
        ffmpeg_exe, '-y', 
        '-i', input_path,
        '-vf', CROP_FILTER,
        '-c:a', 'copy',  # Copy audio without re-encoding to save time
        temp_output
    ]
    
    print(f"Executing video cropping (Parameters: {CROP_FILTER})...")
    try:
        subprocess.run(crop_cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        # Replace original file with the cropped version
        os.replace(temp_output, input_path)
        print(f"Cropping completed successfully!")
    except Exception as e:
        print(f"Error during cropping: {e}")

def download_bilibili_videos(urls, output_dir, ffmpeg_bin_path=None):
    """
    Batch download and crop Bilibili videos using yt-dlp.
    """
    if not check_ffmpeg(ffmpeg_bin_path):
        print("\nError: FFmpeg not detected. Please check FFMPEG_BIN_PATH in the configuration.")
        return

    if not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)

    ydl_opts = {
        'format': 'bestvideo+bestaudio/best',
        'outtmpl': os.path.join(output_dir, '%(title)s.%(ext)s'),
        'quiet': False,
        'no_warnings': True,
        'prefer_ffmpeg': True,
        'merge_output_format': 'mp4',
        'ffmpeg_location': ffmpeg_bin_path,
    }

    try:
        import yt_dlp
    except ImportError:
        print("yt-dlp not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "yt-dlp"])
        import yt_dlp

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        for index, url in enumerate(urls):
            print(f"\n[{index + 1}/{len(urls)}] Processing: {url}")
            try:
                # 1. Download and Merge
                info = ydl.extract_info(url, download=True)
                filename = ydl.prepare_filename(info)
                
                # 2. Perform Cropping if the file exists
                if os.path.exists(filename):
                    crop_video(filename, ffmpeg_bin_path)
                    
            except Exception as e:
                print(f"Task failed: {url}\nError details: {e}")

if __name__ == "__main__":
    print("="*50)
    print("   Bilibili Auto Downloader & Cropper (English Ver)   ")
    print("="*50)
    
    download_bilibili_videos(VIDEO_LIST, output_dir=OUTPUT_DIR, ffmpeg_bin_path=FFMPEG_BIN_PATH)
    
    print("\nAll tasks finished.")
    print("="*50)